# Threshold search for Reject Option Classification

Two searches per dataset:

| Search | Candidates | Objective | Constraint | Result |
|---|---|---|---|---|
| Baseline breaking point| 100 thresholds $t$ (0.01 … 0.99) | balanced accuracy | none | $t_\text{base}$, one threshold for everyone |
| ROC | 100 thresholds $\theta$ × 50 margins $m$ = 5,000 pairs | balanced accuracy | $\lvert\text{DP\_Diff}\rvert \le 0.05$ | $\theta - m$ (unprivileged), $\theta + m$ (privileged) |

The setting for split and classifier used in this notebook is identicial to `ROC_fairness.ipynb`.

In [ ]:
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd

logging.disable(logging.WARNING)      # silence AIF360's notices about optional dependencies
from aif360.algorithms.postprocessing import RejectOptionClassification
from aif360.datasets import AdultDataset, CompasDataset, GermanDataset
from aif360.metrics import BinaryLabelDatasetMetric
logging.disable(logging.NOTSET)

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)
OUT = RESULTS / "roc_thresholds.json"

SEED = 0
TEST_SIZE = 0.30
VAL_SIZE = 0.20

# Search space and constraint: AIF360 defaults, with statistical parity as the fairness constraint
ROC_PARAMS = dict(
    low_class_thresh=0.01, high_class_thresh=0.99, num_class_thresh=100,   # θ and baseline rid 
    num_ROC_margin=50,                                                      # m grid
    metric_name="Statistical parity difference", metric_lb=-0.05, metric_ub=0.05,
)

## Data, split and classifier (as in `ROC_fairness.ipynb`)

In [2]:
def load_adult():
    return AdultDataset(
        protected_attribute_names=["sex"], privileged_classes=[["Male"]],
        categorical_features=["workclass", "education", "marital-status", "occupation",
                              "relationship", "native-country", "race"],        
    )


def load_german():
    return GermanDataset(
        protected_attribute_names=["age"], privileged_classes=[lambda age: age > 25],
        categorical_features=["status", "credit_history", "purpose", "savings", "employment",
                              "other_debtors", "property", "installment_plans", "housing",
                              "skill_level", "telephone", "foreign_worker", "sex"], 
    )


def load_compas():
    return CompasDataset(
        protected_attribute_names=["race"], privileged_classes=[["Caucasian"]],
        categorical_features=["age_cat", "c_charge_degree", "c_charge_desc", "sex"], 
    )


DATASETS = {
    "Adult":  dict(loader=load_adult,  attr="sex",  priv="Male",      unpriv="Female",        fav="income > $50K"),
    "German": dict(loader=load_german, attr="age",  priv="age > 25",  unpriv="age ≤ 25",      fav="good credit risk"),
    "COMPAS": dict(loader=load_compas, attr="race", priv="Caucasian", unpriv="non-Caucasian", fav="no recidivism (2 y)"),
}


def groups(name):
    """AIF360 group definitions: protected attribute 1 = privileged, 0 = unprivileged."""
    a = DATASETS[name]["attr"]
    return dict(unprivileged_groups=[{a: 0}], privileged_groups=[{a: 1}])


datasets = {}
for name, cfg in DATASETS.items():
    ds = cfg["loader"]()
    ds.features = MinMaxScaler().fit_transform(ds.features)   # one-hot columns stay 0/1
    datasets[name] = ds

In [3]:
def stratified_split(ds, seed):
    """Train / validation / test = 50 / 20 / 30 % of rows, stratified on label x protected group."""
    strata = ds.labels.ravel().astype(int) * 2 + ds.protected_attributes[:, 0].astype(int)
    idx = np.arange(len(strata))
    rest, test = train_test_split(idx, test_size=TEST_SIZE, stratify=strata, random_state=seed)
    train, val = train_test_split(rest, test_size=VAL_SIZE / (1 - TEST_SIZE),
                                  stratify=strata[rest], random_state=seed)
    return ds.subset(train), ds.subset(val), ds.subset(test)


def scored(ds, clf, scaler):
    """Copy of `ds` whose .scores hold the classifier's P(favourable | x)."""
    out = ds.copy(deepcopy=True)
    fav_col = list(clf.classes_).index(ds.favorable_label)
    out.scores = clf.predict_proba(scaler.transform(ds.features))[:, [fav_col]]
    return out

## Grid search on the validation set

* **Baseline.** Each $t$ labels validation instances favourable iff $s(x) > t$; the $t$ with the highest balanced accuracy is kept.
* **ROC.** `RejectOptionClassification.fit(validation_true, validation_scored)` runs AIF360's grid search:
  * **Candidates.** 100 $\theta$ (0.01 … 0.99) × 50 $m$ (0 … $\min(\theta, 1 - \theta)$)
  * **Evaluation.** Each pair relabels the validation set based on ROC rule. Balanced accuracy is scored against true labels; DP_Diff on the new labels.
  * **Selection.** Highest balanced accuracy among pairs with $|\text{DP\_Diff}| \le 0.05$.

In [ ]:
def search(name, ds):
    train, val, _ = stratified_split(ds, SEED)

    # classifier 
    std = StandardScaler().fit(train.features)
    clf = LogisticRegression(max_iter=5000).fit(std.transform(train.features), train.labels.ravel())
    val_s = scored(val, clf, std)

    # baseline: highest validation balanced accuracy
    grid = np.linspace(ROC_PARAMS["low_class_thresh"], ROC_PARAMS["high_class_thresh"], ROC_PARAMS["num_class_thresh"])
    y_val, s_val = val.labels.ravel() == val.favorable_label, val_s.scores.ravel()
    bal_acc = [0.5 * ((s_val[y_val] > t).mean() + (s_val[~y_val] <= t).mean()) for t in grid]   # (TPR + TNR) / 2
    t_base = grid[int(np.argmax(bal_acc))]

    # ROC: AIF360's grid search over 100 θ × 50 m, best balanced accuracy subject to |DP_Diff| ≤ 0.05
    roc = RejectOptionClassification(**groups(name), **ROC_PARAMS).fit(val, val_s)
    val_dp = BinaryLabelDatasetMetric(roc.predict(val_s), **groups(name)).statistical_parity_difference()

    return {
        "baseline threshold": float(t_base),
        "ROC threshold θ": float(roc.classification_threshold),
        "ROC margin m": float(roc.ROC_margin),
        "constraint met": bool(ROC_PARAMS["metric_lb"] <= val_dp <= ROC_PARAMS["metric_ub"]),
        "DP_Diff on validation (ROC)": float(val_dp),
        "unprivileged rows in validation": int((val.protected_attributes[:, 0] == 0).sum()),
    }


thresholds = {name: search(name, ds) for name, ds in datasets.items()}

OUT.write_text(json.dumps({
    "settings": {"SEED": SEED, "TEST_SIZE": TEST_SIZE, "VAL_SIZE": VAL_SIZE, "ROC_PARAMS": ROC_PARAMS},
    "thresholds": thresholds,
}, indent=2, ensure_ascii=False))
print(f"saved to {OUT}")
pd.DataFrame(thresholds).round(3)

saved to results/roc_thresholds.json


,Adult,German,COMPAS
baseline threshold,0.198081,0.673232,0.475253
ROC threshold θ,0.188182,0.752424,0.534646
ROC margin m,0.138256,0.090946,0.056982
constraint met,True,True,True
DP_Diff on validation (ROC),-0.046921,0.013562,0.025003
unprivileged rows in validation,2939,38,814
